Importações

In [1]:
import requests
import pandas as pd
import os

# Endereço da API (Swagger)

https://dadosabertos.compras.gov.br/swagger-ui/index.html#/01%20-%20CAT%C3%81LOGO%20-%20MATERIAL/consultarClasseMaterial

URL da API que retornará os dados.



In [2]:
serv_url_classe = "https://dadosabertos.compras.gov.br/modulo-material/2_consultarClasseMaterial"

## Encapsulando a chamada do endpoint em uma função para reaproveitamento

In [3]:
def download(url):
  pagina = 1
  resultado = []
  while True:
      response = requests.get(url, params={'pagina': pagina})
      if response.status_code == 200:
          json_response = response.json()
          print(f"Página {pagina} baixada com sucesso!")
          resultado.extend(json_response["resultado"])

          paginas_restantes = json_response["paginasRestantes"]
          print(f"Páginas restantes: {paginas_restantes}")
          if paginas_restantes == 0:
              break
          pagina += 1
      else:
          print("Erro: ", response.text)
          break # Exit loop on error
  return resultado

PIPELINE ETL
1. Extração
2. Transformação
3. Carga

##1.EXTRAÇÃO
Executando a chamada REST e salvando em staging em formato parquet

In [4]:
resultado = download(serv_url_classe)


Página 1 baixada com sucesso!
Páginas restantes: 1
Página 2 baixada com sucesso!
Páginas restantes: 0


Análise do resultado retornado (JSON)

In [5]:
resultado

[{'codigoClasse': 1005,
  'codigoGrupo': 10,
  'nomeGrupo': 'ARMAMENTO',
  'nomeClasse': 'ARMAS DE FOGO DE CALIBRE ATÉ 120MM',
  'statusClasse': True,
  'dataHoraAtualizacao': '2021-10-16T09:17:13.045775'},
 {'codigoClasse': 1010,
  'codigoGrupo': 10,
  'nomeGrupo': 'ARMAMENTO',
  'nomeClasse': 'ARMAS DE FOGO DE CALIBRE ACIMA DE 30MM ATÉ 75MM',
  'statusClasse': True,
  'dataHoraAtualizacao': '2021-10-16T09:17:13.045775'},
 {'codigoClasse': 1015,
  'codigoGrupo': 10,
  'nomeGrupo': 'ARMAMENTO',
  'nomeClasse': 'ARMAS DE FOGO DE CALIBRE ACIMA DE 75MM ATÉ 125MM',
  'statusClasse': True,
  'dataHoraAtualizacao': '2021-10-16T09:17:13.045775'},
 {'codigoClasse': 1020,
  'codigoGrupo': 10,
  'nomeGrupo': 'ARMAMENTO',
  'nomeClasse': 'ARMAS DE FOGO DE CALIBRE ACIMA DE 125MM ATÉ 150MM',
  'statusClasse': True,
  'dataHoraAtualizacao': '2021-10-16T09:17:13.045775'},
 {'codigoClasse': 1025,
  'codigoGrupo': 10,
  'nomeGrupo': 'ARMAMENTO',
  'nomeClasse': 'ARMAS DE FOGO DE CALIBRE ACIMA DE 150MM 

Aramazenamento do resultado em um data frame.

In [6]:
df_stg_classe = pd.DataFrame(resultado)
df_stg_classe.head(10)


,codigoClasse,codigoGrupo,nomeGrupo,nomeClasse,statusClasse,dataHoraAtualizacao
0,1005,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ATÉ 120MM,True,2021-10-16T09:17:13.045775
1,1010,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 30MM ATÉ 75MM,True,2021-10-16T09:17:13.045775
2,1015,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 75MM ATÉ 125MM,True,2021-10-16T09:17:13.045775
3,1020,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 125MM ATÉ 150MM,True,2021-10-16T09:17:13.045775
4,1025,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 150MM ATÉ 200MM,True,2021-10-16T09:17:13.045775
5,1030,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 200MM ATÉ 300MM,True,2021-10-16T09:17:13.045775
6,1035,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 300MM,True,2021-10-16T09:17:13.045775
7,1040,10,ARMAMENTO,ARMAMENTO E EQUIPAMENTOS DE GUERRA QUÍMICA,True,2021-10-16T09:17:13.045775
8,1045,10,ARMAMENTO,LANÇADORES DE TORPEDOS E DE BOMBAS DE PROFUNDI...,True,2021-10-16T09:17:13.045775
9,1055,10,ARMAMENTO,LANÇADORES DE FOGUETES E PIROTÉCNICOS,True,2021-10-16T09:17:13.045775


In [7]:
df_stg_classe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 711 entries, 0 to 710
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   codigoClasse         711 non-null    int64 
 1   codigoGrupo          711 non-null    int64 
 2   nomeGrupo            711 non-null    object
 3   nomeClasse           711 non-null    object
 4   statusClasse         711 non-null    bool  
 5   dataHoraAtualizacao  711 non-null    object
dtypes: bool(1), int64(2), object(3)
memory usage: 28.6+ KB


Configurar um caminho do Google drive para armazenar os arquivos gerados.

In [8]:
DATA_PATH = '/content/drive/MyDrive/Docencia/IDP/Dados'

Armazenar os dados do dataframe em um arquivo Parquet.

In [9]:
df_stg_classe.to_parquet(os.path.join(DATA_PATH, 'stg_classe.parquet'))

##TRANSFORMAÇÃO
Suponha que a sua organização tem como política formatar os campos do tipo data/hora no formato datetime do python/pandas.

Na nossa etapa de transformação, vamos carregar o parquet da área de staging e fazer uma conversão de formato na data/hora de atualização como string para datetime. Vamos também incluir um campo data carga do tipo data para registrar a data em que a carga está sendo feita.

Primeiro, vamos carregar os dados do arquivo parque para um data frame.

In [10]:
df_classe = pd.read_parquet(os.path.join(DATA_PATH, 'stg_classe.parquet'))

In [11]:
df_classe

,codigoClasse,codigoGrupo,nomeGrupo,nomeClasse,statusClasse,dataHoraAtualizacao
0,1005,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ATÉ 120MM,True,2021-10-16T09:17:13.045775
1,1010,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 30MM ATÉ 75MM,True,2021-10-16T09:17:13.045775
2,1015,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 75MM ATÉ 125MM,True,2021-10-16T09:17:13.045775
3,1020,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 125MM ATÉ 150MM,True,2021-10-16T09:17:13.045775
4,1025,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 150MM ATÉ 200MM,True,2021-10-16T09:17:13.045775
...,...,...,...,...,...,...
706,7120,71,MOBILIÁRIOS,ESTANTES E ARMACOES PARA ALMOXARIFADOS,False,2021-10-16T09:17:13.045775
707,7432,74,"MÁQUINAS PARA ESCRITÓRIO, SISTEMAS DE PROCESSA...",MAQUINAS DE REPROGRAFIA,False,2021-10-16T09:17:13.045775
708,2315,23,VEíCULOS,VEICULOS DE SERVICOS ESPECIAIS,False,2021-10-16T09:17:13.045775
709,6040,60,"MATERIAIS, COMPONENTES, CONJUNTOS E ACESSÓRIOS...",SENSORES DE FIBRA OTICA,False,2021-10-16T09:17:13.045775


Vamos verificar o tipo dos dados do dataframe.

In [12]:
df_classe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 711 entries, 0 to 710
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   codigoClasse         711 non-null    int64 
 1   codigoGrupo          711 non-null    int64 
 2   nomeGrupo            711 non-null    object
 3   nomeClasse           711 non-null    object
 4   statusClasse         711 non-null    bool  
 5   dataHoraAtualizacao  711 non-null    object
dtypes: bool(1), int64(2), object(3)
memory usage: 28.6+ KB


Agora, vamos transformar o tipo do dado da variável dataHoraAtualizacao para o formato datetime do python/pandas.

In [13]:
df_classe['dataHoraAtualizacao'] = pd.to_datetime(df_classe['dataHoraAtualizacao'])

In [14]:
df_classe.head(10)

,codigoClasse,codigoGrupo,nomeGrupo,nomeClasse,statusClasse,dataHoraAtualizacao
0,1005,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ATÉ 120MM,True,2021-10-16 09:17:13.045775
1,1010,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 30MM ATÉ 75MM,True,2021-10-16 09:17:13.045775
2,1015,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 75MM ATÉ 125MM,True,2021-10-16 09:17:13.045775
3,1020,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 125MM ATÉ 150MM,True,2021-10-16 09:17:13.045775
4,1025,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 150MM ATÉ 200MM,True,2021-10-16 09:17:13.045775
5,1030,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 200MM ATÉ 300MM,True,2021-10-16 09:17:13.045775
6,1035,10,ARMAMENTO,ARMAS DE FOGO DE CALIBRE ACIMA DE 300MM,True,2021-10-16 09:17:13.045775
7,1040,10,ARMAMENTO,ARMAMENTO E EQUIPAMENTOS DE GUERRA QUÍMICA,True,2021-10-16 09:17:13.045775
8,1045,10,ARMAMENTO,LANÇADORES DE TORPEDOS E DE BOMBAS DE PROFUNDI...,True,2021-10-16 09:17:13.045775
9,1055,10,ARMAMENTO,LANÇADORES DE FOGUETES E PIROTÉCNICOS,True,2021-10-16 09:17:13.045775


In [15]:
df_classe.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 711 entries, 0 to 710
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   codigoClasse         711 non-null    int64         
 1   codigoGrupo          711 non-null    int64         
 2   nomeGrupo            711 non-null    object        
 3   nomeClasse           711 non-null    object        
 4   statusClasse         711 non-null    bool          
 5   dataHoraAtualizacao  711 non-null    datetime64[ns]
dtypes: bool(1), datetime64[ns](1), int64(2), object(2)
memory usage: 28.6+ KB


##CARGA
Agora vamos armazenar os dados do novo data set em um parquet.

In [16]:
df_classe.to_parquet(os.path.join(DATA_PATH, 'classe_transformed.parquet'))
print(f"DataFrame df_classe salvo como 'classe_transformed.parquet' em {DATA_PATH}")

DataFrame df_classe salvo como 'classe_transformed.parquet' em /content/drive/MyDrive/Docencia/IDP/Dados
